# Import Data to MySQL

**ใช้ notebook นี้หลังจากสร้าง database + tables แล้ว**

จะ import:
1. ETF Master (50 rows)
2. Benchmark Portfolios (35 rows)
3. Benchmark Holdings (117 rows)
4. Price History (208,700 rows)

## Step 0: Configuration

In [ ]:
import mysql.connector
from mysql.connector import Error
import pandas as pd
import os

# MySQL Configuration (แก้ password ถ้าต้องการ)
MYSQL_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': 'krittanut123456',
    'database': 'portfolio_backtesting'
}

print("✅ Configuration loaded")
print(f"   Database: {MYSQL_CONFIG['database']}")

# ============================================================
# วิธีที่ 1: ให้ระบุ path เอง (แก้บรรทัดด้านล่าง)
# ============================================================
# ถ้ารู้ path แน่ชัด ให้ uncomment บรรทัดนี้และใส่ path ของคุณ:
# PROJECT_ROOT = r'C:\Users\YourName\desktop-tutorial'  # Windows
# PROJECT_ROOT = '/Users/YourName/desktop-tutorial'      # Mac/Linux

# ============================================================
# วิธีที่ 2: ให้หาอัตโนมัติ
# ============================================================
PROJECT_ROOT = None

print("\n🔍 Detecting project directory...")
print(f"   Current directory: {os.getcwd()}")

# Check current directory first
if os.path.exists('data/etf_list.csv'):
    PROJECT_ROOT = os.getcwd()
    print(f"   ✅ Found data files in current directory!")
else:
    # Search upwards
    print("   🔍 Searching for project root...")
    current = os.getcwd()
    
    for level in range(10):  # Search up to 10 levels
        if os.path.exists(os.path.join(current, 'data', 'etf_list.csv')):
            PROJECT_ROOT = current
            print(f"   ✅ Found project root at: {current}")
            break
        
        parent = os.path.dirname(current)
        if parent == current:  # Reached filesystem root
            break
        current = parent

# Change to project directory if found
if PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)
    print(f"\n✅ Working directory set to: {os.getcwd()}")
else:
    print("\n⚠️  Could not auto-detect project root")
    print("\n📋 แก้ไข:")
    print("   1. ปิด notebook นี้")
    print("   2. เปิด Terminal/CMD")
    print("   3. cd ไปที่โฟลเดอร์ desktop-tutorial")
    print("   4. รัน: jupyter notebook import_data.ipynb")
    print("\n   หรือ:")
    print("   1. แก้บรรทัดที่ 17 ในcell นี้")
    print("   2. ใส่ path ของ desktop-tutorial")
    print("   3. รัน cell นี้ใหม่")
    
# Verify files
print("\n📁 Checking required files...")
required_files = [
    'data/etf_list.csv',
    'data/benchmark_portfolios.csv', 
    'data/benchmark_holdings.csv',
    'data/etf_price_history.csv'
]

files_status = {}
for filepath in required_files:
    exists = os.path.exists(filepath)
    status = "✅" if exists else "❌"
    print(f"   {status} {filepath}")
    files_status[filepath] = exists

# Summary
if all(files_status.values()):
    print("\n✅ All files ready! You can proceed to next steps.")
elif not files_status['data/etf_price_history.csv']:
    print("\n⚠️  Missing: etf_price_history.csv")
    print("   Run this command first:")
    print("   python scripts/generate_sample_data.py")
else:
    print("\n⚠️  Some files are missing. Please check the file paths.")

## Step 1: Connect to MySQL

In [ ]:
# Connect to MySQL
try:
    connection = mysql.connector.connect(**MYSQL_CONFIG)
    cursor = connection.cursor(dictionary=True)
    print(f"✅ Connected to MySQL: {MYSQL_CONFIG['database']}")
    
    # Check tables exist
    cursor.execute("SHOW TABLES")
    tables = [list(row.values())[0] for row in cursor.fetchall()]
    print(f"\n📊 Found {len(tables)} tables:")
    for table in tables:
        print(f"   - {table}")
        
    if len(tables) < 9:
        print("\n⚠️  Warning: ไม่ครบ 9 tables")
        print("   กรุณารัน complete_setup.sql ก่อน")
        
except Error as e:
    print(f"❌ Error: {e}")
    print("\nกรุณาตรวจสอบ:")
    print("  1. MySQL กำลังรันอยู่หรือไม่?")
    print("  2. Password ถูกต้องหรือไม่?")
    print("  3. Database 'portfolio_backtesting' มีอยู่แล้วหรือไม่?")
    raise

## Step 2: Import ETF Master (50 ETFs)

In [ ]:
print("="*70)
print("📥 Import ETF Master Data")
print("="*70)

# Read CSV
etf_df = pd.read_csv('data/etf_list.csv')
print(f"\nFound {len(etf_df)} ETFs in file")
display(etf_df.head())

# Clear existing data
cursor.execute("DELETE FROM etf_master")
connection.commit()
print("\n✅ Cleared existing data")

# Import
count = 0
for _, row in etf_df.iterrows():
    try:
        cursor.execute("""
            INSERT INTO etf_master
            (ticker_symbol, etf_name, asset_class, region, sector, expense_ratio, inception_date)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (
            row['ticker_symbol'],
            row['etf_name'],
            row['asset_class'],
            row.get('region'),
            row.get('sector'),
            float(row['expense_ratio']) if pd.notna(row.get('expense_ratio')) else None,
            row.get('inception_date') if pd.notna(row.get('inception_date')) else None
        ))
        count += 1
    except Error as e:
        print(f"⚠️  Error {row['ticker_symbol']}: {e}")

connection.commit()
print(f"\n✅ Imported {count} ETFs")

# Verify
cursor.execute("SELECT COUNT(*) as count FROM etf_master")
total = cursor.fetchone()['count']
print(f"   Total in database: {total}")

## Step 3: Import Benchmark Portfolios (35 portfolios)

In [ ]:
print("="*70)
print("📥 Import Benchmark Portfolios")
print("="*70)

# Read CSV
benchmark_df = pd.read_csv('data/benchmark_portfolios.csv')
print(f"\nFound {len(benchmark_df)} benchmarks in file")
display(benchmark_df.head())

# Clear existing data
cursor.execute("DELETE FROM benchmark_portfolios")
connection.commit()
print("\n✅ Cleared existing data")

# Import
count = 0
for _, row in benchmark_df.iterrows():
    try:
        cursor.execute("""
            INSERT INTO benchmark_portfolios
            (benchmark_name, description, risk_level, target_return, asset_allocation)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            row['benchmark_name'],
            row.get('description'),
            row.get('risk_level', 'Moderate'),
            float(row['target_return']) if pd.notna(row.get('target_return')) else None,
            row.get('asset_allocation')
        ))
        count += 1
    except Error as e:
        print(f"⚠️  Error {row['benchmark_name']}: {e}")

connection.commit()
print(f"\n✅ Imported {count} benchmark portfolios")

# Verify
cursor.execute("SELECT COUNT(*) as count FROM benchmark_portfolios")
total = cursor.fetchone()['count']
print(f"   Total in database: {total}")

## Step 4: Import Benchmark Holdings (117 holdings)

In [ ]:
print("="*70)
print("📥 Import Benchmark Holdings")
print("="*70)

# Get mappings
cursor.execute("SELECT benchmark_id, benchmark_name FROM benchmark_portfolios")
benchmark_map = {row['benchmark_name']: row['benchmark_id'] for row in cursor.fetchall()}

cursor.execute("SELECT etf_id, ticker_symbol FROM etf_master")
etf_map = {row['ticker_symbol']: row['etf_id'] for row in cursor.fetchall()}

# Read CSV
holdings_df = pd.read_csv('data/benchmark_holdings.csv')
print(f"\nFound {len(holdings_df)} holdings in file")
display(holdings_df.head())

# Clear existing data
cursor.execute("DELETE FROM benchmark_holdings")
connection.commit()
print("\n✅ Cleared existing data")

# Import
count = 0
errors = 0
for _, row in holdings_df.iterrows():
    benchmark_id = benchmark_map.get(row['benchmark_name'])
    etf_id = etf_map.get(row['ticker_symbol'])
    
    if not benchmark_id or not etf_id:
        errors += 1
        continue
    
    try:
        cursor.execute("""
            INSERT INTO benchmark_holdings
            (benchmark_id, etf_id, target_weight)
            VALUES (%s, %s, %s)
        """, (benchmark_id, etf_id, float(row['target_weight'])))
        count += 1
    except Error as e:
        print(f"⚠️  Error: {e}")
        errors += 1

connection.commit()
print(f"\n✅ Imported {count} benchmark holdings")
if errors > 0:
    print(f"⚠️  Skipped {errors} rows")

# Verify
cursor.execute("SELECT COUNT(*) as count FROM benchmark_holdings")
total = cursor.fetchone()['count']
print(f"   Total in database: {total}")

## Step 5: Import Price History (208,700 rows) ⏱️ 1-2 นาที

In [ ]:
print("="*70)
print("📥 Import Price History (ใช้เวลา 1-2 นาที)")
print("="*70)

# Check file exists
if not os.path.exists('data/etf_price_history.csv'):
    print("❌ ไม่พบไฟล์: data/etf_price_history.csv")
    print("ℹ️  กรุณารัน: python scripts/generate_sample_data.py ก่อน")
else:
    # Get ETF mapping
    cursor.execute("SELECT etf_id, ticker_symbol FROM etf_master")
    etf_map = {row['ticker_symbol']: row['etf_id'] for row in cursor.fetchall()}
    
    # Clear existing data
    cursor.execute("DELETE FROM price_history")
    connection.commit()
    print("\n✅ Cleared existing data")
    
    # Import in chunks
    chunk_size = 10000
    total_imported = 0
    
    print("\nImporting...")
    
    for chunk in pd.read_csv('data/etf_price_history.csv', chunksize=chunk_size):
        # Map ticker to etf_id
        chunk['etf_id'] = chunk['ticker_symbol'].map(etf_map)
        chunk = chunk.dropna(subset=['etf_id'])
        chunk['etf_id'] = chunk['etf_id'].astype(int)
        
        # Prepare batch
        batch = []
        for _, row in chunk.iterrows():
            batch.append((
                int(row['etf_id']),
                row['date'],
                float(row['open']),
                float(row['high']),
                float(row['low']),
                float(row['close']),
                float(row['adj_close']),
                int(float(row['volume']))
            ))
        
        # Batch insert
        if batch:
            try:
                cursor.executemany("""
                    INSERT INTO price_history
                    (etf_id, date, open, high, low, close, adj_close, volume)
                    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                """, batch)
                connection.commit()
                total_imported += len(batch)
                print(f"   Imported {total_imported:,} records...", end='\r')
            except Error as e:
                print(f"\n⚠️  Error: {e}")
    
    print(f"\n\n✅ Imported {total_imported:,} price records")
    
    # Verify
    cursor.execute("SELECT COUNT(*) as count FROM price_history")
    total = cursor.fetchone()['count']
    print(f"   Total in database: {total:,}")

## Summary: ตรวจสอบข้อมูลทั้งหมด

In [ ]:
print("="*70)
print("IMPORT SUMMARY".center(70))
print("="*70)
print()

# Get all tables
cursor.execute("SHOW TABLES")
tables = [list(row.values())[0] for row in cursor.fetchall()]

# Count rows in each table
summary_data = []
for table in tables:
    cursor.execute(f"SELECT COUNT(*) as count FROM {table}")
    count = cursor.fetchone()['count']
    summary_data.append({'Table': table, 'Rows': count})
    print(f"   {table:30s} {count:>10,} rows")

# Create summary DataFrame
summary_df = pd.DataFrame(summary_data)

print("\n" + "="*70)
print("✅ Import เสร็จสมบูรณ์!".center(70))
print("="*70)
print("\nพร้อมใช้งาน! เปิด main.ipynb หรือรัน: python main.py")

# Display as DataFrame
print("\n")
display(summary_df)

## Close Connection (Optional)

In [ ]:
# Uncomment to close connection
# cursor.close()
# connection.close()
# print("👋 Database connection closed")